### Parameter study: Learn ODE for parachute jump
Use PINNs to solve the ODE modelling the fall with a parachute for different "friction parameters" $D\in [0.01, 1]$:
\begin{align*}
    \partial_t^2 u(t) &= D(\partial_t u(t))^2 - g, \text{ \, \,  for  } t\in [0, 3] \\
    u(0) &= H \\
    \partial_t u(0) &= 0
\end{align*}
H=50 is the (fixed) initial height, g=9.81 is the gravitation constant.

In [ ]:
# This block is for GPU selection. Please execute.
import pathlib
import os
user = int(str(pathlib.Path().resolve())[24:26])
os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [ ]:
import torch
import torchphysics as tp
import pytorch_lightning as pl

### TODO: Set all parameters Here all parameters are defined:
t_min, t_max = ...
D_min, D_max = ...
g = ...
H = ...

# Number of samples drawn for the ode and initial conditions.
N_ode = 5000
N_initial = 100

train_iterations = 5000
learning_rate = 1.e-2

In [ ]:
### TODO: Implement the spaces
T = ...
U = ...
D = ...

### TODO: Define the time interval 
time_interval = ...
D_interval = ...

### TODO: Create random unifom samplers for the ode condition and for the initial conditions
ode_sampler = ...
initial_sampler = ...

In [ ]:
### TODO: Create a fully-connected neural network with two layers which each have 20 hidden neurons
model = ...

In [ ]:
### TODO: Define condition for the ODE:
def ode_residual(...):
    return ...

ode_condition = ...

In [ ]:
### TODO: Define condition for the initial position:
def position_residual(...):
    return ...

initial_position_condition = ...

In [ ]:
### TODO: Define condition for the initial velocity:
def velocity_residual(...):
    return ...

initial_velocity_condition = ...

In [ ]:
### The optimizer is already implemented. Just fill in the conditions that should be optimized and you can start the training.

optim = tp.OptimizerSetting(optimizer_class=torch.optim.Adam, lr=learning_rate)
solver = tp.solver.Solver(train_conditions=[ode_condition, initial_position_condition, initial_velocity_condition], optimizer_setting=optim)

trainer = pl.Trainer(devices=1, accelerator="gpu",
                     max_steps=train_iterations,
                     logger=False,
                     benchmark=True,
                     enable_checkpointing=False)

trainer.fit(solver)

In [ ]:
### Here, we plot the analytical solution and the (pointwise) error curve for some fixed test value for D:
### TODO: Inspect the plots for different values of D_test
import matplotlib.pyplot as plt

D_test = 0.05

# This is the analytical solution of the ODE
def analytic_solution(t, D):
    return 1/D * (-torch.log((1+torch.exp(-2*torch.sqrt(D*g)*t))/2) - torch.sqrt(D*g)*t) + H

plot_sampler = tp.samplers.PlotSampler(time_interval, 100, data_for_other_variables={'D': D_test})
fig = tp.utils.plot(model, lambda u: u, plot_sampler)
plt.title("predicted solution")

plot_sampler = tp.samplers.PlotSampler(time_interval, 100, data_for_other_variables={'D': D_test})
fig = tp.utils.plot(model, lambda t,D: analytic_solution(t, D), plot_sampler,)
plt.title("analytical solution")

fig = tp.utils.plot(model, lambda u,t,D: torch.abs(u - analytic_solution(t, D)), plot_sampler)
plt.title("absolute error")

In [ ]:
# Let's manually plot the model's prediction and analytic solution in the same figure

ts = torch.linspace(0, 3, 100)
D_test = 0.01

# Create tensor of (t,D)-points with constant value for D, namely D=D_test
points = torch.zeros(len(ts), 2)
points[:, 0] = ts
points[:, 1] = D_test

# Compute and plot true solution
solution = torch.tensor([analytic_solution(*point) for point in points])
plt.plot(ts, solution, label='analytic solution')

# Transfer pytorch tensors into a TorchPhysics `tp.spaces.Points` objects, in order to evaluate the TorchPhysics model on these points.
points = tp.spaces.Points(points, T*D)
prediction = model(points).as_tensor.detach().flatten() # as_tensor: creates a pytorch tensor, detach: removes it from GPU, flatten: tensor has dimension 1 required for plotting 
plt.plot(ts, prediction, label='prediction')

plt.legend()
plt.title(f'D = {D_test}')
plt.show()

In [ ]:
# TODO: Compute the mean squared error between the prediction and solution
# TODO: Check the quality of the prediction for D outside the training interval
# TODO: To improve the prediction, train the model further with smaller learning rates
mse = torch.mean((prediction - solution)**2)
print(f'Mean squared error for D={D_test} is {mse}')

In [ ]:
# BONUS: Two options a) Open the notebook inverse.ipynb for learning how to solve a parameter identification problem with PINNs
#                    b) Learn the parachute ODE above also for different gravitiy constants g (so for two variable parameters D, g. You may need to increase the size of the model.)